# GTEx global alignment

Two analyses per GTEx tissue, each testing whether the selected SHAP LV(s) recover the true tissue by ORA against the GTEx tissue gene-set database:

1. **top1**: the single highest-ranked SHAP LV per tissue.
2. **cumulative25**: the top-ranked LVs needed to reach `CUMULATIVE_PCT`% of cumulative SHAP per tissue, the same criterion used in `00_LV_importance_true_labels.ipynb` / `01_LV_importance_true_labels_biology.ipynb`.


In [1]:
library(here)
library(dplyr)
library(tidyr)
library(stringr)
library(Matrix)
library(clusterProfiler)


SHAP_DIR <- here('output', '03_model_biology', '01_gtex',
                 '03_rf_true_labels', '00_LV_importance_true_labels',
                 'gtex_feature_importance_true_labels_binary_shap')
OUT_DIR  <- here('output', '03_model_biology', '01_gtex', '03_rf_true_labels', '02_global_alignment')
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

CLAMP_RDS      <- here('output', '01_model_building', '02_gtex', '01_CLAMP', 'CLAMPfull.rds')
GTEX_TISSUE_DB <- here('data', 'archs4', 'GTEx_Tissues_pathMat.rds')

N_LVS_PER_TISSUE <- 1L
CUMULATIVE_PCT   <- 25  # same threshold as 01_LV_importance_true_labels_biology.ipynb
TOP_GENE_PCT     <- 0.01
FDR_THRESH       <- 0.05


here() starts at /home/marc/Documents/pivlab/clamp-analyses




Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





Attaching package: ‘Matrix’




The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

S Xu, E Hu, Y Cai, Z Xie, X Luo, L Zhan, W Tang, Q Wang, B Liu, R Wang,
W Xie, T Wu, L Xie, G Yu. Using clusterProfiler to characterize
multiomics data. Nature Protocols. 2024, 19(11):3292-3320




Attaching package: ‘clusterProfiler’




The following object is masked from ‘package:stats’:

    filter




In [2]:
shap_all <- read.delim(file.path(SHAP_DIR, 'all_shap_positive.tsv'),
                       stringsAsFactors = FALSE, check.names = FALSE)

selected_lvs_top1 <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::slice_head(n = N_LVS_PER_TISSUE) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selected_lvs_cum <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::mutate(
        reaches_thresh = Cumulative_Percent >= CUMULATIVE_PCT,
        cutoff_rank = if (any(reaches_thresh)) min(Rank[reaches_thresh]) else max(Rank)
    ) %>%
    dplyr::filter(Rank <= cutoff_rank) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selections <- list(top1 = selected_lvs_top1, cumulative25 = selected_lvs_cum)

cat('Tissues:', dplyr::n_distinct(selected_lvs_top1$Tissue), '\n')
for (nm in names(selections)) {
    cat(sprintf('  [%s] selected LV/tissue rows: %d, unique LVs: %d\n',
                nm, nrow(selections[[nm]]), dplyr::n_distinct(selections[[nm]]$LV)))
}

dplyr::bind_rows(lapply(names(selections), function(nm) {
    selections[[nm]] %>%
        dplyr::count(Tissue, name = 'n_selected_lvs') %>%
        dplyr::mutate(analysis = nm)
})) %>%
    tidyr::pivot_wider(names_from = analysis, values_from = n_selected_lvs) %>%
    dplyr::arrange(Tissue)


Tissues: 49 


  [top1] selected LV/tissue rows: 49, unique LVs: 41
  [cumulative25] selected LV/tissue rows: 226, unique LVs: 124


Tissue,top1,cumulative25
<chr>,<int>,<int>
Adipose - Subcutaneous,1,6
Adipose - Visceral (Omentum),1,5
Adrenal Gland,1,2
Artery - Aorta,1,3
Artery - Coronary,1,4
Artery - Tibial,1,4
Brain - Amygdala,1,4
Brain - Anterior cingulate cortex (BA24),1,6
Brain - Caudate (basal ganglia),1,8


In [3]:
clamp <- readRDS(CLAMP_RDS)
Z_full <- as.matrix(clamp$Z)
rm(clamp)

all_selected_lvs <- unique(unlist(lapply(selections, function(df) df$LV)))
selected_unique_lvs <- intersect(all_selected_lvs, colnames(Z_full))
missing_lvs <- setdiff(all_selected_lvs, colnames(Z_full))
if (length(missing_lvs) > 0) warning('Missing LVs in Z: ', paste(missing_lvs, collapse = ', '))

Z <- Z_full[, selected_unique_lvs, drop = FALSE]
rm(Z_full)
universe_genes <- rownames(Z)
n_top_genes <- max(1L, ceiling(TOP_GENE_PCT * nrow(Z)))

cat('Z:', nrow(Z), 'genes x', ncol(Z), 'unique LVs (union across analyses)\n')
cat('Top genes per LV:', n_top_genes, '\n')


Z: 21613 genes x 124 unique LVs (union across analyses)


Top genes per LV: 217 


In [4]:
gtex_mat <- readRDS(GTEX_TISSUE_DB)
term_names <- unname(colnames(gtex_mat))

parse_gtex_tissue <- function(x) {
    x <- sub('^GTEx_Tissues_', '', x)
    x <- sub(' (Male|Female) [0-9]+-[0-9]+ Up$', '', x)
    x
}

normalize_text <- function(x) {
    x <- tolower(x)
    x <- gsub('[^a-z0-9]+', ' ', x)
    stringr::str_squish(x)
}

term2gene <- lapply(seq_along(term_names), function(i) {
    rownames(gtex_mat)[gtex_mat[, i] != 0]
})
names(term2gene) <- term_names

term2gene_df <- do.call(rbind, lapply(names(term2gene), function(term) {
    data.frame(term = term, gene = term2gene[[term]], stringsAsFactors = FALSE)
}))

term_map <- data.frame(
    term = term_names,
    parsed_tissue = parse_gtex_tissue(term_names),
    normalized_tissue = normalize_text(parse_gtex_tissue(term_names)),
    stringsAsFactors = FALSE
)

tissue_check <- dplyr::bind_rows(selections) %>%
    dplyr::distinct(Tissue) %>%
    dplyr::mutate(
        normalized_tissue = normalize_text(Tissue),
        matched_in_gtex_db = normalized_tissue %in% term_map$normalized_tissue,
        n_db_sets = vapply(normalized_tissue, function(x) sum(term_map$normalized_tissue == x), integer(1))
    )

cat('GTEx tissue gene sets:', length(term2gene), '\n')
stopifnot(all(tissue_check$matched_in_gtex_db))
tissue_check


GTEx tissue gene sets: 511 


Tissue,normalized_tissue,matched_in_gtex_db,n_db_sets
<chr>,<chr>,<lgl>,<int>
Adipose - Subcutaneous,adipose subcutaneous,TRUE,12
Adipose - Visceral (Omentum),adipose visceral omentum,TRUE,12
Adrenal Gland,adrenal gland,TRUE,11
Artery - Aorta,artery aorta,TRUE,12
Artery - Coronary,artery coronary,TRUE,11
Artery - Tibial,artery tibial,TRUE,12
Brain - Amygdala,brain amygdala,TRUE,8
Brain - Anterior cingulate cortex (BA24),brain anterior cingulate cortex ba24,TRUE,9
Brain - Caudate (basal ganglia),brain caudate basal ganglia,TRUE,10


In [5]:
run_gtex_tissue_ora <- function(genes, universe) {
    res <- tryCatch(
        clusterProfiler::enricher(
            gene          = genes,
            universe      = universe,
            TERM2GENE     = term2gene_df,
            pAdjustMethod = 'BH',
            pvalueCutoff  = 1,
            qvalueCutoff  = 1,
            minGSSize     = 10,
            maxGSSize     = 500
        ),
        error = function(e) NULL
    )
    if (is.null(res)) return(NULL)
    df <- as.data.frame(res)
    if (nrow(df) == 0) return(NULL)
    df
}

top_genes_per_lv <- lapply(colnames(Z), function(lv) {
    vals <- Z[, lv]
    universe_genes[order(vals, decreasing = TRUE)[seq_len(n_top_genes)]]
})
names(top_genes_per_lv) <- colnames(Z)

ora_list <- lapply(names(top_genes_per_lv), function(lv) {
    df <- run_gtex_tissue_ora(top_genes_per_lv[[lv]], universe_genes)
    if (is.null(df)) return(NULL)
    df$LV <- lv
    df
})

ora_all <- do.call(rbind, Filter(Negate(is.null), ora_list))
if (is.null(ora_all)) {
    ora_all <- data.frame()
} else {
    rownames(ora_all) <- NULL
    ora_all <- ora_all %>%
        dplyr::left_join(term_map, by = c('ID' = 'term')) %>%
        dplyr::select(LV, ID, Description, parsed_tissue, normalized_tissue,
                      GeneRatio, BgRatio, pvalue, p.adjust, qvalue, geneID, Count)
}

write.csv(ora_all, file.path(OUT_DIR, 'gtex_tissue_ora_per_lv.csv'), row.names = FALSE)
cat('ORA rows:', nrow(ora_all), '\n')
head(ora_all)


ORA rows: 32823 


,LV,ID,Description,parsed_tissue,normalized_tissue,GeneRatio,BgRatio,pvalue,p.adjust,qvalue,geneID,Count
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<int>
1,LV263,GTEx_Tissues_Adipose - Subcutaneous Female 20-29 Up,GTEx_Tissues_Adipose - Subcutaneous Female 20-29 Up,Adipose - Subcutaneous,adipose subcutaneous,44/155,98/6121,4.397540e-46,1.319262e-43,1.222053e-43,CIDEC/CIDEA/ADIPOQ/PLIN1/LGALS12/GPD1/LEP/GPAM/LIPE/GYG2/LPL/SCD/G0S2/PLIN4/SLC19A3/KLB/AQP7/SLC7A10/PDE3B/FABP4/PPARG/SLC29A4/MEST/ACVR1C/CEBPA/AGPAT2/GPBAR1/LVRN/AKR1C2/PNPLA2/NMB/PRKAR2B/CD36/TMEM132C/ACACB/RETSAT/CD300LG/RDH5/AIFM2/FAH/FAM89A/PECR/ABCD2/RBP7,44
2,LV263,GTEx_Tissues_Adipose - Subcutaneous Male 20-29 Up,GTEx_Tissues_Adipose - Subcutaneous Male 20-29 Up,Adipose - Subcutaneous,adipose subcutaneous,42/155,99/6121,1.274868e-42,1.912302e-40,1.771396e-40,CIDEC/CIDEA/ADIPOQ/PLIN1/LGALS12/GPD1/GPAM/LIPE/LPL/SCD/G0S2/PLIN4/SLC19A3/KLB/PRRT4/AQP7/PDE3B/FABP4/FASN/PPARG/WNT11/MEST/ACVR1C/CEBPA/AGPAT2/LVRN/AKR1C2/TIMP4/PNPLA2/TNMD/MGST1/PRKAR2B/CD36/TMEM132C/ACACB/RETSAT/CD300LG/AIFM2/FAM89A/ELMOD3/DTX1/RBP7,42
3,LV263,GTEx_Tissues_Adipose - Subcutaneous Female 40-49 Up,GTEx_Tissues_Adipose - Subcutaneous Female 40-49 Up,Adipose - Subcutaneous,adipose subcutaneous,41/155,95/6121,5.818140e-42,5.818140e-40,5.389435e-40,CIDEC/CIDEA/ADIPOQ/PLIN1/LGALS12/GPD1/LEP/LIPE/GYG2/LPL/G0S2/PLIN4/SLC19A3/KLB/AQP7/SLC7A10/PDE3B/FABP4/PPARG/SLC29A4/MEST/ACVR1C/CEBPA/AGPAT2/GPBAR1/LVRN/AKR1C2/TIMP4/PNPLA2/TNMD/MGST1/PRKAR2B/CD36/TMEM132C/ACACB/RETSAT/CD300LG/RDH5/AIFM2/FAM89A/ELMOD3,41
4,LV263,GTEx_Tissues_Adipose - Visceral (Omentum) Female 20-29 Up,GTEx_Tissues_Adipose - Visceral (Omentum) Female 20-29 Up,Adipose - Visceral (Omentum),adipose visceral omentum,40/155,97/6121,6.273229e-40,4.704922e-38,4.358243e-38,CIDEC/CIDEA/ADIPOQ/PLIN1/LGALS12/GPD1/GPAM/LIPE/GYG2/LPL/G0S2/SLC19A3/MRAP/KLB/PRRT4/AQP7/DGAT2/SLC7A10/FASN/PPARG/MEST/AGPAT2/GPBAR1/TIMP4/PNPLA2/NMB/PRKAR2B/TMEM132C/ACSL1/ACACB/RETSAT/RDH5/ALDH1L1/PC/NEURL2/AIFM2/FAH/FAM89A/HAS1/MDFI,40
5,LV263,GTEx_Tissues_Adipose - Visceral (Omentum) Female 40-49 Up,GTEx_Tissues_Adipose - Visceral (Omentum) Female 40-49 Up,Adipose - Visceral (Omentum),adipose visceral omentum,32/155,64/6121,2.756060e-35,1.653636e-33,1.531789e-33,ADIPOQ/THRSP/LGALS12/GPAM/GYG2/LPL/SCD/PRRT4/AQP7/DGAT2/SLC7A10/FABP4/FASN/PPARG/MEST/AGPAT2/TIMP4/PNPLA2/NMB/PRKAR2B/ITLN1/CD36/PRG4/ELOVL6/ALOX15/AIFM2/UPK3B/FAM89A/HAS1/CD209/MSLN/ME1,32
6,LV263,GTEx_Tissues_Adipose - Subcutaneous Female 30-39 Up,GTEx_Tissues_Adipose - Subcutaneous Female 30-39 Up,Adipose - Subcutaneous,adipose subcutaneous,34/155,95/6121,2.396600e-31,1.198300e-29,1.110004e-29,CIDEC/CIDEA/ADIPOQ/PLIN1/GPD1/LEP/LIPE/GYG2/LPL/PLIN4/SLC19A3/KLB/AQP7/PDE3B/FABP4/PPARG/ACVR1C/AGPAT2/LVRN/TIMP4/PNPLA2/TNMD/PRKAR2B/CD36/TMEM132C/ACACB/RETSAT/CD300LG/RDH5/AIFM2/FAH/ELMOD3/ZNF117/ABCD2,34


In [6]:
run_alignment_summary <- function(selected_lvs, ora_all, out_dir) {
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
    sig_ora <- ora_all %>% dplyr::filter(LV %in% selected_lvs$LV, p.adjust < FDR_THRESH)

    detail <- selected_lvs %>%
        dplyr::rowwise() %>%
        dplyr::mutate(
            normalized_true_tissue = normalize_text(Tissue),
            n_sig_terms = sum(sig_ora$LV == LV),
            n_true_terms = sum(sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue),
            tissue_correct = n_true_terms > 0,
            best_any_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            best_true_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            matched_terms = paste(sig_ora$ID[sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue], collapse = ' | ')
        ) %>%
        dplyr::ungroup()

    tissue_summary <- detail %>%
        dplyr::group_by(Tissue) %>%
        dplyr::summarise(
            n_selected_lvs = dplyr::n(),
            tissue_correct = any(tissue_correct),
            correct_lvs = paste(LV[tissue_correct], collapse = ';'),
            best_true_padj = if (all(is.na(best_true_padj))) NA_real_ else min(best_true_padj, na.rm = TRUE),
            .groups = 'drop'
        ) %>%
        dplyr::mutate(correct_score = as.integer(tissue_correct))

    final_pct_tissue_correct <- 100 * mean(tissue_summary$tissue_correct)
    final_summary <- data.frame(
        n_tissues = nrow(tissue_summary),
        n_tissues_correct = sum(tissue_summary$tissue_correct),
        pct_tissue_correct = final_pct_tissue_correct,
        stringsAsFactors = FALSE
    )

    write.csv(detail, file.path(out_dir, 'gtex_global_alignment_detail.csv'), row.names = FALSE)
    write.csv(tissue_summary, file.path(out_dir, 'gtex_global_alignment_summary.csv'), row.names = FALSE)
    write.csv(final_summary, file.path(out_dir, 'gtex_global_alignment_final_pct.csv'), row.names = FALSE)

    list(detail = detail, tissue_summary = tissue_summary, final_summary = final_summary)
}

results <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], ora_all, file.path(OUT_DIR, nm))
})
names(results) <- names(selections)

dplyr::bind_rows(lapply(names(results), function(nm) {
    results[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))


analysis,n_tissues,n_tissues_correct,pct_tissue_correct
<chr>,<int>,<int>,<dbl>
top1,49,43,87.7551
cumulative25,49,49,100.0000


In [7]:
cat('--- top1 ---\n')
results$top1$tissue_summary %>% dplyr::arrange(Tissue)

cat('--- cumulative25 ---\n')
results$cumulative25$tissue_summary %>% dplyr::arrange(Tissue)


--- top1 ---


Tissue,n_selected_lvs,tissue_correct,correct_lvs,best_true_padj,correct_score
<chr>,<int>,<lgl>,<chr>,<dbl>,<int>
Adipose - Subcutaneous,1,TRUE,LV263,1.319262e-43,1
Adipose - Visceral (Omentum),1,TRUE,LV263,4.704922e-38,1
Adrenal Gland,1,TRUE,LV183,8.202318e-63,1
Artery - Aorta,1,TRUE,LV145,4.240123e-11,1
Artery - Coronary,1,TRUE,LV232,4.098348e-03,1
Artery - Tibial,1,TRUE,LV563,3.743852e-05,1
Brain - Amygdala,1,TRUE,LV136,6.030062e-16,1
Brain - Anterior cingulate cortex (BA24),1,TRUE,LV136,5.723977e-16,1
Brain - Caudate (basal ganglia),1,TRUE,LV34,5.273289e-53,1


--- cumulative25 ---


Tissue,n_selected_lvs,tissue_correct,correct_lvs,best_true_padj,correct_score
<chr>,<int>,<lgl>,<chr>,<dbl>,<int>
Adipose - Subcutaneous,6,TRUE,LV263;LV369;LV5;LV460;LV106;LV68,1.319262e-43,1
Adipose - Visceral (Omentum),5,TRUE,LV263;LV460;LV151;LV127;LV106,4.704922e-38,1
Adrenal Gland,2,TRUE,LV183;LV3,8.202318e-63,1
Artery - Aorta,3,TRUE,LV145;LV531;LV37,1.966264e-45,1
Artery - Coronary,4,TRUE,LV232;LV106;LV40;LV99,6.061080e-12,1
Artery - Tibial,4,TRUE,LV563;LV90;LV37;LV387,7.034525e-35,1
Brain - Amygdala,4,TRUE,LV136;LV121;LV188;LV194,3.109316e-38,1
Brain - Anterior cingulate cortex (BA24),6,TRUE,LV136;LV24;LV121;LV2;LV85;LV164,1.650848e-56,1
Brain - Caudate (basal ganglia),8,TRUE,LV34;LV51;LV44;LV328;LV155;LV230;LV54;LV169,5.273289e-53,1


In [8]:
detail_cols <- c('Tissue', 'LV', 'Rank', 'Mean_SHAP_Tissue', 'Cumulative_Percent',
                 'tissue_correct', 'n_sig_terms', 'n_true_terms',
                 'best_true_padj', 'matched_terms')

cat('--- top1 ---\n')
results$top1$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)

cat('--- cumulative25 ---\n')
results$cumulative25$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)


--- top1 ---


Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Adipose - Subcutaneous,LV263,1,0.05842972,6.549590,TRUE,30,12,1.319262e-43,GTEx_Tissues_Adipose - Subcutaneous Female 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Male 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Female 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Female 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Female 60-69 Up | GTEx_Tissues_Adipose - Subcutaneous Female 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Male 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Female 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Male 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Male 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Male 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Male 60-69 Up
Adipose - Visceral (Omentum),LV263,1,0.06652897,7.420449,TRUE,30,12,4.704922e-38,GTEx_Tissues_Adipose - Visceral (Omentum) Female 20-29 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 40-49 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 30-39 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 50-59 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 70-79 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 40-49 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 50-59 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 30-39 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 60-69 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 20-29 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 60-69 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 70-79 Up
Adrenal Gland,LV183,1,0.14297289,14.913246,TRUE,11,11,8.202318e-63,GTEx_Tissues_Adrenal Gland Male 40-49 Up | GTEx_Tissues_Adrenal Gland Male 50-59 Up | GTEx_Tissues_Adrenal Gland Male 20-29 Up | GTEx_Tissues_Adrenal Gland Female 20-29 Up | GTEx_Tissues_Adrenal Gland Female 60-69 Up | GTEx_Tissues_Adrenal Gland Male 60-69 Up | GTEx_Tissues_Adrenal Gland Female 40-49 Up | GTEx_Tissues_Adrenal Gland Male 30-39 Up | GTEx_Tissues_Adrenal Gland Female 30-39 Up | GTEx_Tissues_Adrenal Gland Female 50-59 Up | GTEx_Tissues_Adrenal Gland Male 70-79 Up
Artery - Aorta,LV145,1,0.14942737,15.904544,TRUE,20,11,4.240123e-11,GTEx_Tissues_Artery - Aorta Female 20-29 Up | GTEx_Tissues_Artery - Aorta Male 20-29 Up | GTEx_Tissues_Artery - Aorta Female 30-39 Up | GTEx_Tissues_Artery - Aorta Male 40-49 Up | GTEx_Tissues_Artery - Aorta Male 30-39 Up | GTEx_Tissues_Artery - Aorta Female 40-49 Up | GTEx_Tissues_Artery - Aorta Male 70-79 Up | GTEx_Tissues_Artery - Aorta Female 50-59 Up | GTEx_Tissues_Artery - Aorta Female 60-69 Up | GTEx_Tissues_Artery - Aorta Male 50-59 Up | GTEx_Tissues_Artery - Aorta Male 60-69 Up
Artery - Coronary,LV232,1,0.08830223,9.612166,TRUE,24,7,4.098348e-03,GTEx_Tissues_Artery - Coronary Male 50-59 Up | GTEx_Tissues_Artery - Coronary Male 60-69 Up | GTEx_Tissues_Artery - Coronary Male 30-39 Up | GTEx_Tissues_Artery - Coronary Female 50-59 Up | GTEx_Tissues_Artery - Coronary Male 40-49 Up | GTEx_Tissues_Artery - Coronary Male 70-79 Up | GTEx_Tissues_Artery - Coronary Male 20-29 Up
Artery - Tibial,LV563,1,0.07735856,8.249628,TRUE,14,11,3.743852e-05,GTEx_Tissues_Artery - Tibial Female 30-39 Up | GTEx_Tissues_Artery - Tibial Male 20-29 Up | GTEx_Tissues_Artery - Tibial Female 40-49 Up | GTEx_Tissues_Artery - Tibial Male 40-49 Up | GTEx_Tissues_Artery - Tibial Female 20-29 Up | GTEx_Tissues_Artery - Tibial Male 30-39 Up | GTEx_Tissues_Artery - Tibial Female 70-79 Up | GTEx_Tissues_Artery - Tibial Male 60-69 Up | GTEx_Tissues_Artery - Tibial Male 50-59 Up | GTEx_Tissues_Artery - Tibial Male 70-79 Up | GTEx_Tissues_Artery - Tibial Female 50-59 Up
Brain - Amygdala,LV136,1,0.11450020,12.766679,TRUE,40,8,6.030062e-16,GTEx_Tissues_Brain - Amygdala Female 40-49 Up | GTEx_Tissues_Brain - Amygdala Male 40-49 Up | GTEx_Tissues_Brain - Amygdala Male 20-29 Up | GTEx_Tissues_Brain - Amygdala Female 6

--- cumulative25 ---


Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Adipose - Subcutaneous,LV263,1,0.05842972,6.549590,TRUE,30,12,1.319262e-43,GTEx_Tissues_Adipose - Subcutaneous Female 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Male 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Female 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Female 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Female 60-69 Up | GTEx_Tissues_Adipose - Subcutaneous Female 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Male 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Female 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Male 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Male 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Male 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Male 60-69 Up
Adipose - Subcutaneous,LV369,2,0.04485311,11.577330,TRUE,35,12,5.596882e-10,GTEx_Tissues_Adipose - Subcutaneous Male 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Male 60-69 Up | GTEx_Tissues_Adipose - Subcutaneous Female 60-69 Up | GTEx_Tissues_Adipose - Subcutaneous Male 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Male 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Male 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Female 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Female 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Female 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Female 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Male 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Female 20-29 Up
Adipose - Subcutaneous,LV5,3,0.03934853,15.988043,FALSE,29,0,NA,
Adipose - Subcutaneous,LV460,4,0.03714234,20.151456,TRUE,32,12,2.481387e-22,GTEx_Tissues_Adipose - Subcutaneous Male 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Female 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Female 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Male 30-39 Up | GTEx_Tissues_Adipose - Subcutaneous Male 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Female 60-69 Up | GTEx_Tissues_Adipose - Subcutaneous Female 50-59 Up | GTEx_Tissues_Adipose - Subcutaneous Female 20-29 Up | GTEx_Tissues_Adipose - Subcutaneous Male 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Male 40-49 Up | GTEx_Tissues_Adipose - Subcutaneous Female 70-79 Up | GTEx_Tissues_Adipose - Subcutaneous Male 60-69 Up
Adipose - Subcutaneous,LV106,5,0.03528964,24.107193,FALSE,0,0,NA,
Adipose - Subcutaneous,LV68,6,0.03014824,27.486614,FALSE,6,0,NA,
Adipose - Visceral (Omentum),LV263,1,0.06652897,7.420449,TRUE,30,12,4.704922e-38,GTEx_Tissues_Adipose - Visceral (Omentum) Female 20-29 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 40-49 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 30-39 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 50-59 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 70-79 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 40-49 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 50-59 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 30-39 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 60-69 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 20-29 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 60-69 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 70-79 Up
Adipose - Visceral (Omentum),LV460,2,0.04684646,12.645568,TRUE,32,12,5.681395e-20,GTEx_Tissues_Adipose - Visceral (Omentum) Male 50-59 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 40-49 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 60-69 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 70-79 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 30-39 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 40-49 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 50-59 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 30-39 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Female 20-29 Up | GTEx_Tissues_Adipose - Visceral (Omentum) Male 20-29 Up | GTEx_Tissues_Adipose - Visceral (Oment

In [9]:
# Per-LV correctness within the cumulative25 selection (not just "any LV correct")
per_lv_summary <- results$cumulative25$detail %>%
    dplyr::group_by(Tissue) %>%
    dplyr::summarise(
        n_lvs = dplyr::n(),
        n_correct = sum(tissue_correct),
        pct_correct = 100 * n_correct / n_lvs,
        .groups = "drop"
    ) %>%
    dplyr::arrange(pct_correct)

print(per_lv_summary, n = Inf)

total_lvs <- sum(per_lv_summary$n_lvs)
total_correct <- sum(per_lv_summary$n_correct)
cat(sprintf(
    "\nOverall: %d/%d LV-tissue rows correct (%.1f%%) across cumulative25 selection\n",
    total_correct, total_lvs, 100 * total_correct / total_lvs
))
cat(sprintf("Tissues at 100%% LV concordance: %d/%d\n",
            sum(per_lv_summary$pct_correct == 100), nrow(per_lv_summary)))

write.csv(per_lv_summary, file.path(OUT_DIR, "cumulative25", "gtex_global_alignment_per_lv_pct.csv"), row.names = FALSE)



# A tibble: 49 × 4
   Tissue                                    n_lvs n_correct pct_correct
   <chr>                                     <int>     <int>       <dbl>
 1 Pituitary                                     5         1        20  
 2 Brain - Cerebellar Hemisphere                 9         2        22.2
 3 Cells - Cultured fibroblasts                  4         1        25  
 4 Whole Blood                                   4         1        25  
 5 Esophagus - Gastroesophageal Junction         6         2        33.3
 6 Heart - Left Ventricle                        6         2        33.3
 7 Muscle - Skeletal                             3         1        33.3
 8 Spleen                                        3         1        33.3
 9 Testis                                        3         1        33.3
10 Brain - Substantia nigra                      8         3        37.5
11 Cells - EBV-transformed lymphocytes           5         2        40  
12 Adipose - Subcutaneous       


Overall: 137/226 LV-tissue rows correct (60.6%) across cumulative25 selection


Tissues at 100% LV concordance: 13/49


In [10]:
stopifnot(file.exists(file.path(OUT_DIR, 'gtex_tissue_ora_per_lv.csv')))

for (nm in names(selections)) {
    out_dir <- file.path(OUT_DIR, nm)
    expected_rows <- nrow(selections[[nm]])
    stopifnot(nrow(results[[nm]]$detail) == expected_rows)
    stopifnot(nrow(results[[nm]]$tissue_summary) == dplyr::n_distinct(shap_all$Tissue))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_detail.csv')))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_summary.csv')))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_final_pct.csv')))
    cat(sprintf(
        '[%s] Checks passed. Denominator = %d tissues and %d selected LV/tissue rows. Final %% tissue correct = %.2f\n',
        nm, nrow(results[[nm]]$tissue_summary), nrow(results[[nm]]$detail),
        results[[nm]]$final_summary$pct_tissue_correct
    ))
}


[top1] Checks passed. Denominator = 49 tissues and 49 selected LV/tissue rows. Final % tissue correct = 87.76
[cumulative25] Checks passed. Denominator = 49 tissues and 226 selected LV/tissue rows. Final % tissue correct = 100.00
